# AutoLyap: Gradient method

This notebook uses AutoLyap to analyze the gradient method with an iteration-independent Lyapunov certificate.


## Setup

Install AutoLyap. The notebook defaults to
`backend="cvxpy", cvxpy_solver="CLARABEL"`.


In [ ]:
%pip install -q autolyap

## Problem setup

Consider the unconstrained minimization problem

$$
\underset{x \in \mathcal{H}}{\text{minimize}}\ f(x),
$$

where $\mathcal{H}$ is a real Hilbert space, and
$f : \mathcal{H} \to \mathbb{R}$ is $\mu$-strongly convex and $L$-smooth,
with $0 < \mu < L$.

Equivalently, we may write the associated inclusion problem

$$
\text{find } x \in \mathcal{H} \;\text{ such that }\; 0 \in \partial f(x) = \{\nabla f(x)\}.
$$

For an initial point $x^0 \in \mathcal{H}$ and step size $0 < \gamma < 2/L$, the
gradient update is

$$
(\forall k \in \mathbb{N})\quad
x^{k+1} = x^k - \gamma \nabla f(x^k).
$$

We use AutoLyap to search for the contraction factor
$\rho \in [0,1)$ with the smallest value among those it certifies such that

$$
\|x^k - x^\star\|^2 \in \mathcal{O}(\rho^k) \quad \text{ as } \quad k\to\infty,
$$

where $x^\star \in \operatorname{Argmin}_{x \in \mathcal{H}} f(x)$.


In [ ]:
from autolyap import SolverOptions
from autolyap.algorithms import GradientMethod
from autolyap.iteration_independent import IterationIndependent
from autolyap.problemclass import InclusionProblem, SmoothStronglyConvex

## Define problem data and create instances

Choose the problem parameters and create an instance of the inclusion problem and gradient method.


In [ ]:
mu, L, gamma = 1.0, 4.0, 0.2

problem = InclusionProblem([SmoothStronglyConvex(mu, L)])
algorithm = GradientMethod(gamma=gamma)

## Iteration-Independent Lyapunov analysis

The goal is to search for parameters $Q, q, S, s$ such that the
iteration-independent quadratic Lyapunov conditions hold along every trajectory
of the method over the problem class:

$$
\mathcal{V}(Q,q,k+1) \le \rho\,\mathcal{V}(Q,q,k) - \mathcal{R}(S,s,k) \tag{C1}
$$

$$
\mathcal{V}(Q,q,k) \ge \mathcal{V}(P,p,k) \ge 0 \tag{C2}
$$

$$
\mathcal{R}(S,s,k) \ge \mathcal{R}(T,t,k) \ge 0 \tag{C3}
$$

where $P, p, T, t$ are user specified parameters.

When $0\leq\rho<1$, `(C1)` and `(C2)` give directly

$$
0 \le \mathcal{V}(P,p,k) \le \mathcal{V}(Q,q,k) \le \rho^k\mathcal{V}(Q,q,0)
\xrightarrow[k\to\infty]{} 0.
$$

We will pick $P$ and $p$ such that $\mathcal{V}(P,p,k)=\|x^k-x^\star\|^2$ to get the convergence conclusion of interest, i.e.,

$$
\|x^k-x^\star\|^2 \in \mathcal{O}(\rho^k) \quad \text{ as } \quad k\to\infty.
$$


## Choose the targets $P, p, T, t$ 

The helper
`IterationIndependent.LinearConvergence.get_parameters_distance_to_solution`
returns $P, p, T, t$, which allows us to fix the target expressions to
$$
\mathcal{V}(P,p,k)=\|x^k-x^\star\|^2  \quad \text{and} \quad  \mathcal{R}(T,t,k)= 0.
$$

In [ ]:
P, p, T, t = (
    IterationIndependent.LinearConvergence.get_parameters_distance_to_solution(
        algorithm
    )
)

## Do the Lyapunov search
In the search call below, we choose `S_equals_T=True` and `s_equals_t=True`, so that
$$
\mathcal{R}(S,s,k)=\mathcal{R}(T,t,k).
$$
We also choose `remove_C3=True`, which removes `(C3)` as a separate constraint because it becomes redundant under that identification.

Then `bisection_search_rho` varies $\rho$ to estimate the smallest feasible contraction factor for this chosen target.

In [ ]:
search_result = IterationIndependent.LinearConvergence.bisection_search_rho(
    problem,
    algorithm,
    P,
    T,
    p=p,
    t=t,
    S_equals_T=True,
    s_equals_t=True,
    remove_C3=True,
    lower_bound=0.0, # rho_lower_bound
    upper_bound=1.0, # rho_upper_bound
    solver_options=SolverOptions(backend="cvxpy", cvxpy_solver="CLARABEL"),
    verbosity=0,
)

## Compare with theory

The theoretical comparison, from
[Polyak (1963)](https://doi.org/10.1016/0041-5553(63)90382-3), is

$$
\|x^k - x^\star\|^2 \in \mathcal{O}(\rho^k) \quad \text{ as } \quad k\to\infty,
$$

where 
$$
\rho = \max\{|1-\gamma L|,\;|1-\gamma\mu|\}^2
\quad
\text{and}
\quad
x^\star \in \operatorname{Argmin}_{x \in \mathcal{H}} f(x).
$$

In [ ]:
if search_result["status"] != "feasible":
    raise RuntimeError(
        "No feasible Lyapunov certificate in the requested rho interval."
    )

rho_theory = max(abs(1.0 - gamma * L), abs(1.0 - gamma * mu)) ** 2

print(f"status:       {search_result['status']}")
print(f"solve_status: {search_result['solve_status']}")
print(f"rho (AutoLyap): {float(search_result['rho']):.8f}")
print(f"rho (theory):   {rho_theory:.8f}")

## Sweeping over the step size $\gamma$

Sweeping over 100 values of $\gamma$ on the open interval $0 < \gamma < 2/L$ gives the plot below, with the theoretical rate in black and AutoLyap certificates as blue dots.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

gamma_theory_values = np.linspace(0.0, 2.0 / L, 102)
gamma_values = gamma_theory_values[1:-1]
rho_theory_values = np.maximum(
    np.abs(1.0 - L * gamma_theory_values),
    np.abs(1.0 - mu * gamma_theory_values),
) ** 2

sweep_algorithm = GradientMethod(gamma=float(gamma_values[0]))
P_sweep, p_sweep, T_sweep, t_sweep = (
    IterationIndependent.LinearConvergence.get_parameters_distance_to_solution(
        sweep_algorithm
    )
)
sweep_solver_options = SolverOptions(backend="cvxpy", cvxpy_solver="CLARABEL")

rho_autolyap_values = []
for idx, gamma_value in enumerate(gamma_values, start=1):
    sweep_algorithm.set_gamma(float(gamma_value))
    result = IterationIndependent.LinearConvergence.bisection_search_rho(
        problem,
        sweep_algorithm,
        P_sweep,
        T_sweep,
        p=p_sweep,
        t=t_sweep,
        S_equals_T=True,
        s_equals_t=True,
        remove_C3=True,
        lower_bound=0.0,
        upper_bound=1.0,
        solver_options=sweep_solver_options,
        verbosity=0,
    )
    if result["status"] != "feasible":
        raise RuntimeError(
            f"No feasible Lyapunov certificate for gamma={gamma_value:.6f}."
        )
    rho_autolyap_values.append(float(result["rho"]))

    if idx == 1 or idx % 10 == 0 or idx == len(gamma_values):
        print(
            f"Solved {idx:>3}/{len(gamma_values)}"
        )

rho_autolyap_values = np.array(rho_autolyap_values)

fig, ax = plt.subplots(figsize=(6, 3), dpi=160, constrained_layout=True)
ax.plot(gamma_theory_values, rho_theory_values, color="black", linewidth=2.0, label=r"$\max\left\{|1-\gamma L|,\,|1-\gamma \mu|\right\}^2$")
ax.scatter(gamma_values, rho_autolyap_values, s=24, color="#1f77b4", label="AutoLyap")
ax.set_xlabel(r"$\gamma$")
ax.set_ylabel(r"$\rho$", rotation=0)
ax.set_title(rf"Gradient method: contraction factor vs step size ($L={L:g},\ \mu={mu:g}$)")
ax.set_xlim(0.0, 2.0 / L)
ax.set_ylim(0.3, 1.0)
ax.grid(True, alpha=0.3)
ax.legend()
plt.show()

## References

- [Polyak, B. T. (1963)](https://doi.org/10.1016/0041-5553(63)90382-3).
  *Gradient methods for the minimisation of functionals*. USSR Computational
  Mathematics and Mathematical Physics, 3(4), 864-878.
